# CSV → Final Cut Pro XML

실행 순서: **1 → 2 → 3 → 4 → 5**

입력 파일은 다음 두 종류이며 서로 다른 위치에 저장됩니다.

```text
my-video/
├── timeline.csv   ← CSV 또는 timeline.xlsx 중 하나
└── Media/         ← CSV에 적은 사진과 영상
```

2번: CSV/Excel · 3번: 사진/영상

## 1. 실행 코드 준비

코드, FFmpeg, Excel 변환 패키지를 설치합니다.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo = Path('/content/csv-to-fcpxml-starter')
if (repo / '.git').is_dir():
    # 셀을 다시 실행한 경우 기존 폴더를 지우지 않고 최신 코드만 받습니다.
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run([
        'git', 'clone',
        'https://github.com/Kongdataif/csv-to-fcpxml-starter.git',
        str(repo),
    ], check=True)

os.chdir(repo)
if shutil.which('ffprobe') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'준비 완료: {repo}')

## 2. CSV 또는 Excel 한 개 업로드

- 선택: `.csv` 또는 `.xlsx` 한 개
- 저장: `my-video/timeline.csv` 또는 `my-video/timeline.xlsx`
- XLSX 변환 결과: `my-video/output/timeline_converted.csv`
- Numbers: **파일 > 다음으로 내보내기 > Excel** · [Apple 안내](https://support.apple.com/guide/numbers/export-to-excel-or-another-file-format-tan3b922d4ad/mac)
- `파일` 열: 3번에서 올릴 미디어 파일명

In [ ]:
from google.colab import files
from pathlib import Path

project = Path('my-video')
project.mkdir(parents=True, exist_ok=True)

# 업로드 창에서 CSV 또는 XLSX 기획표 한 개만 선택합니다.
plan_upload = files.upload()
if len(plan_upload) != 1:
    raise ValueError('CSV 또는 XLSX 기획표를 정확히 한 개만 선택해주세요.')

plan_name, plan_data = next(iter(plan_upload.items()))
suffix = Path(plan_name).suffix.lower()
if suffix == '.numbers':
    raise ValueError('Numbers에서 파일 > 다음으로 내보내기 > Excel을 선택한 뒤 XLSX를 올려주세요.')
if suffix not in {'.csv', '.xlsx'}:
    raise ValueError(f'CSV 또는 XLSX 파일이 아닙니다: {plan_name}')

# 샘플이나 이전 실행의 다른 형식을 제거해 입력이 두 개가 되지 않게 합니다.
for old_input in (project / 'timeline.csv', project / 'timeline.xlsx'):
    old_input.unlink(missing_ok=True)
plan_target = project / f'timeline{suffix}'
plan_target.write_bytes(plan_data)
print(f'기획표 저장 완료: {plan_name} → {plan_target}')

## 3. 사진·영상 업로드

- 선택: 기획표에 적은 사진·영상 전체
- 저장: `my-video/Media/`
- 파일명: 기획표의 `파일` 값과 확장자까지 일치
- 영상: MOV, MP4, M4V, MKV, AVI · 사진: JPG, JPEG, PNG, HEIC, TIF, TIFF

In [ ]:
from google.colab import files
from pathlib import Path

media_dir = Path('my-video/Media')
media_dir.mkdir(parents=True, exist_ok=True)
supported = {'.mov', '.mp4', '.m4v', '.mkv', '.avi', '.jpg', '.jpeg', '.png', '.heic', '.tif', '.tiff'}

# 업로드 창에서 기획표에 적은 사진과 영상을 모두 선택합니다.
media_upload = files.upload()
invalid = [name for name in media_upload if Path(name).suffix.lower() not in supported]
if invalid:
    raise ValueError('지원하지 않는 파일입니다: ' + ', '.join(invalid))
if not media_upload:
    raise ValueError('사진이나 영상을 한 개 이상 선택해주세요.')

guide_file = media_dir / '여기에_사진과_영상을_넣으세요.txt'
guide_file.unlink(missing_ok=True)
for name, data in media_upload.items():
    (media_dir / Path(name).name).write_bytes(data)

print(f'미디어 {len(media_upload)}개 저장 완료: {media_dir}')
for path in sorted(media_dir.iterdir()):
    print(' -', path.name)

## 4. 출력 설정 후 변환

| 파라미터 | 값 | 의미 |
|---|---|---|
| `LAYOUT` | `portrait` | 세로 9:16, 1080×1920 |
| `LAYOUT` | `landscape` | 가로 16:9, 1920×1080 |
| `LAYOUT` | `both` | 세로와 가로를 모두 생성 |
| `FIT` | `fit` | 원본 전체 표시. 비율이 다르면 여백 가능 |
| `FIT` | `fill` | 화면을 채움. 원본 가장자리 잘림 가능 |

설정 후 실행합니다. 결과: `my-video/output/`

In [ ]:
from pathlib import Path
import subprocess

LAYOUT = 'portrait'  # @param ['portrait', 'landscape', 'both']
FIT = 'fit'  # @param ['fit', 'fill']

# 선택한 값만 명령어 인자로 전달합니다.
subprocess.run([
    'python3', 'run.py', 'my-video',
    '--layout', LAYOUT,
    '--fit', FIT,
], check=True)

print('생성 결과:')
for path in sorted(Path('my-video/output').glob('*.fcpxml')):
    print(' -', path)

## 5. 결과 ZIP 다운로드

FCPXML과 미디어를 ZIP으로 다운로드합니다. ZIP 크기는 원본 미디어 크기에 따라 달라집니다.

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive('/content/fcpxml-result', 'zip', 'my-video')
print(f'다운로드 준비 완료: {archive}')
files.download(archive)